## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

load_dotenv(override=True)

True

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [ ]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
message = "Latest AI Agent frameworks in 2026"

result = await Runner.run(search_agent, message)
display(Markdown(result.final_output))

In 2026, several AI agent frameworks have emerged, each catering to specific needs:

- **LangGraph**: Ideal for complex, stateful workflows, offering full control and debugging tools. ([bootstraparena.com](https://bootstraparena.com/blog/top-10-ai-agent-frameworks-2026?utm_source=openai))

- **OpenAI Agents SDK**: Best suited for projects within the OpenAI ecosystem, providing seamless integration with OpenAI models. ([bootstraparena.com](https://bootstraparena.com/blog/top-10-ai-agent-frameworks-2026?utm_source=openai))

- **CrewAI**: Excels in multi-agent team collaboration, featuring role-based agents and built-in collaboration tools. ([bootstraparena.com](https://bootstraparena.com/blog/top-10-ai-agent-frameworks-2026?utm_source=openai))

- **AutoGen**: Developed by Microsoft, it specializes in multi-agent conversations, supporting custom agent topologies and enterprise-grade observability. ([aimagicx.com](https://www.aimagicx.com/blog/best-open-source-ai-agent-frameworks-2026?utm_source=openai))

- **MCP**: Recognized for its simplicity and ease of use, making it suitable for straightforward applications. ([leaper.dev](https://leaper.dev/blog/ai-agent-frameworks-2026?utm_source=openai))

Additionally, the formation of the Nemotron Coalition by Nvidia, comprising eight AI companies, aims to co-develop open frontier models, enhancing the capabilities of AI agents. ([tomshardware.com](https://www.tomshardware.com/tech-industry/artificial-intelligence/nvidias-nemoclaw-coalition-brings-eight-ai-labs-together-to-build-open-frontier-models?utm_source=openai))

As AI agents become integral to enterprise operations, selecting the appropriate framework is crucial, considering factors like project complexity, team expertise, and specific use cases. 

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [ ]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")

    # These 2 fields will be used later on in 
    # search(item: WebSearchItem) function


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

    # This field will be called later on in 
    # perform_searches(search_plan: WebSearchPlan) function

planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
) # output a list of web searches item

In [ ]:

message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find the most recent frameworks developed or released for AI agents in the year 2026.', query='latest AI agent frameworks 2026'), WebSearchItem(reason='To understand the trends in artificial intelligence agent development, including popular frameworks, tools, and technologies.', query='AI agent development trends 2026'), WebSearchItem(reason='To gather information on specific frameworks that have gained traction in the AI community for 2026.', query='top AI agent frameworks 2026')]


In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("minhtanbuinguyen@gmail.com")
    to_email = To("tan.m.bui96@outlook.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [ ]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1149ea340>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [ ]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)



In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

<div style="font-size: 0.85em;">

### `Field` in Pydantic: With vs Without Description

`Field` comes from **Pydantic** and adds a `description` to each field. This description is embedded in the JSON schema sent to the model, so the LLM knows exactly what to write in each field when generating structured output.

---

#### With `Field(description=...)`

```json
{
  "short_summary": {
    "type": "string",
    "description": "A short 2-3 sentence summary of the findings."
  },
  "markdown_report": {
    "type": "string",
    "description": "The final report"
  },
  "follow_up_questions": {
    "type": "array",
    "items": {"type": "string"},
    "description": "Suggested topics to research further"
  }
}
```

#### Without `Field(description=...)`

```json
{
  "short_summary": {"type": "string"},
  "markdown_report": {"type": "string"},
  "follow_up_questions": {"type": "array", "items": {"type": "string"}}
}
```

---

#### What that means in practice

| Field | Without description (model guesses) | With description (model is guided) |
|---|---|---|
| `short_summary` | Might write a full paragraph or just one word | Knows to write exactly 2–3 sentences |
| `markdown_report` | Might return plain text or bullet points | Knows this is the full detailed report |
| `follow_up_questions` | Might return full sentences instead of topics | Knows these should be concise question strings |

The descriptions act as **per-field system prompts** — the more specific they are, the more predictable and correct the model's structured output will be.

</div>

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [ ]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches] 
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

<div style="font-size: 0.85em;">

### Search Pipeline: `plan_searches`, `perform_searches`, `search`

These three async functions form the **search planning and execution pipeline** in the Deep Research agent workflow.

---

#### `plan_searches(query)`
Takes the user's research query (e.g. `"Latest AI Agent frameworks in 2025"`) and hands it to `planner_agent`, which uses structured outputs to return a `WebSearchPlan` — a list of `WebSearchItem` objects, each containing a search `query` string and a `reason` explaining why that search is relevant. It essentially breaks one broad research question into several focused web searches.

---

#### `perform_searches(search_plan)`
Takes the `WebSearchPlan` produced above and **runs all the searches concurrently**. It creates an `asyncio` task for each `WebSearchItem` and fires them all in parallel with `asyncio.gather()`. This is the **parallelisation pattern** — instead of running N searches sequentially (slow), it runs them all at once and waits for all results together.

---

#### `search(item)`
The worker function called for each individual `WebSearchItem`. It formats the search term and its reason into a prompt, sends it to `search_agent` (which uses `WebSearchTool` to actually query the web), and returns the summarised text output. This is what each parallel task in `perform_searches` actually executes.

---

### How they chain together

```
plan_searches(query)
    └─► planner_agent → WebSearchPlan (list of N search items)

perform_searches(search_plan)
    ├─► search(item[0])  ─┐
    ├─► search(item[1])   ├─ all run in parallel via asyncio.gather
    └─► search(item[2])  ─┘
            each └─► search_agent + WebSearchTool → summarised text
```

The results then feed into `write_report()`, which passes all the summaries to `writer_agent` to synthesise the final report.

</div>

In [ ]:
# The following hidden cell explains perform_searches(search_plan: WebSearchPlan)
# Reveal to read

<div style="font-size: 0.85em;">

### `perform_searches` — Step-by-Step Breakdown

```python
async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results
```

---

#### **Parameter: `search_plan: WebSearchPlan`**

A Pydantic model passed in — it has a `.searches` field which is a **list of `WebSearchItem` objects** (each with a `query` string and a `reason`). This was produced by `plan_searches()` in the step before.

---

#### **Line 1: List comprehension → `tasks`**

```python
tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
```

- Loops through every `WebSearchItem` in the plan
- For each `item`, calls `search(item)` — but **does NOT await it yet**
- `asyncio.create_task()` wraps each coroutine and **schedules it to run concurrently immediately**
- **Returns:** a `list` of `Task` objects (all searches now running in the background simultaneously)

> All searches **start at the same time** at this point — not one after another.

**Example `tasks` value** — for query `"Latest AI Agent frameworks in 2026"` (from the cell above):

```python
tasks = [
    <Task pending coro=<search() running>>,  # query: 'latest AI agent frameworks 2026'
    <Task pending coro=<search() running>>,  # query: 'AI agent development trends 2026'
    <Task pending coro=<search() running>>,  # query: 'top AI agent frameworks 2026'
]
# 3 asyncio.Task objects — all fire at the same time
```

---

#### **Line 2: `await asyncio.gather(*tasks)` → `results`**

```python
results = await asyncio.gather(*tasks)
```

- `*tasks` unpacks the list as individual arguments to `gather()`
- `asyncio.gather()` **waits for ALL tasks to finish**
- Returns results **in the same order as the input tasks** (not the order they finish)
- **Returns:** a `list[str]` — one summarised search result per `WebSearchItem`

**Example `results` value** — what `asyncio.gather()` returns:

```python
results = [
    # results[0] — from query 'latest AI agent frameworks 2026'
    "In 2026, several AI agent frameworks have emerged. LangGraph is ideal for "
    "complex stateful workflows. OpenAI Agents SDK integrates seamlessly with "
    "OpenAI models. CrewAI excels at multi-agent collaboration with role-based agents...",

    # results[1] — from query 'AI agent development trends 2026'
    "Key trends in 2026 include hierarchical multi-agent systems, tool-use "
    "standardisation via MCP, and growing enterprise adoption of autonomous pipelines. "
    "Nvidia's Nemotron Coalition brings 8 AI labs together to co-develop open models...",

    # results[2] — from query 'top AI agent frameworks 2026'
    "Most adopted in 2026: LangGraph leads for production deployments with full "
    "control and debugging. CrewAI popular for role-based crews. AutoGen dominant "
    "in research. MCP noted for simplicity in straightforward applications..."
]
# list[str] — 3 strings, one per search, in the SAME ORDER as search_plan.searches
```

---

#### **Return value**

```python
return results  # list[str] — passed directly into write_report() as search_results
```

---

#### Why `asyncio.gather` instead of a `for` loop?

| Approach | Time for 3 searches (1s each) |
|---|---|
| Sequential `for` loop with `await` | ~3 seconds |
| `asyncio.gather()` (parallel) | ~1 second |

All N searches run **in parallel** instead of waiting for one to finish before starting the next.

</div>

### The next 2 functions write a report and email it

In [ ]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [ ]:
query ="Latest AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")




Starting research...
Planning searches...
Will perform 3 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>